# Graph RAG Question Generator

This notebook generates a high-quality test question set for evaluating Graph RAG agents. It supports loading content from **local files** (Text, PDF, Markdown), **directories**, and **URLs**.

In [4]:
import os
import json
import requests
from typing import List
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import pypdf
from bs4 import BeautifulSoup

# Load API Key from .env
load_dotenv(dotenv_path='../.env')

if "GOOGLE_API_KEY" not in os.environ:
    print("⚠️ GOOGLE_API_KEY not found in environment. Please check your .env file.")
else:
    GEMINI_API=os.environ.get("GOOGLE_API_KEY")
    print("✅ Environment loaded successfully.")

# Load Guidelines
with open('TEST_SET_GUIDELINES.md', 'r') as f:
    guidelines = f.read()

print("✅ Guidelines Loaded successfully.")

✅ Environment loaded successfully.
✅ Guidelines Loaded successfully.


## 1. Define Sources
Add your local file paths, directory paths, or URLs here.

In [5]:
def get_links_from_html(html_file):
    links = []
    with open(html_file, 'r') as f:
        soup = BeautifulSoup(f, 'html.parser')
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        # Filter for Google Drive links and PDF files
        if href.startswith('https://drive.google.com/file/d/') or href.endswith('.pdf'):
            links.append(href)
    return links

original_sources = ["../Knowledge/"]
sources = original_sources

SUPPORTED_EXTENSIONS = {'.pdf', '.txt', '.md', '.json', '.html'}

def load_file_content(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()
    if ext not in SUPPORTED_EXTENSIONS:
        return ""
    try:
        if ext == '.pdf':
            reader = pypdf.PdfReader(file_path)
            return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
        else:
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
    except Exception as e:
        print(f"  ⚠️ Error reading {file_path}: {e}")
    return ""

def load_content(source: str) -> str:
    if source.startswith(('http://', 'https://')):
        print(f"Fetching URL: {source}")
        try:
            response = requests.get(source)
            response.raise_for_status()
            # This is a bit of a hack to get the content from google drive links
            # as it might require authentication or dealing with different file types.
            # For now, we will just return the text of the response.
            return response.text
        except Exception as e:
            print(f"  ❌ Error fetching {source}: {e}")
            return ""

    if os.path.isdir(source):
        print(f"Scanning folder: {source}")
        contents = []
        for root, dirs, files in os.walk(source):
            # Skip hidden directories
            dirs[:] = [d for d in dirs if not d.startswith('.')]
            for file in sorted(files):
                if file.startswith('.'):
                    continue
                full_path = os.path.join(root, file)
                ext = os.path.splitext(file)[1].lower()
                if ext not in SUPPORTED_EXTENSIONS:
                    print(f"  ⏭ Skipping unsupported file: {file}")
                    continue
                content = load_file_content(full_path)
                if content:
                    print(f"  ✅ Loaded: {os.path.relpath(full_path, source)}")
                    contents.append(content)
                else:
                    print(f"  ⚠️ Empty or unreadable: {file}")
        print(f"  → {len(contents)} file(s) loaded from {source}")
        return "\n\n---\n\n".join(contents)

    elif os.path.isfile(source):
        print(f"Reading file: {source}")
        return load_file_content(source)

    else:
        print(f"❌ Source not found: {source}")
        return ""

combined_text = "\n\n---\n\n".join([load_content(s) for s in sources if s])
print(f"\nTotal characters loaded: {len(combined_text)}")

Scanning folder: ../Knowledge/
  ✅ Loaded: BodyParagraphs.pdf
  ⏭ Skipping unsupported file: Parser_And_Evaluator.ipynb
  → 1 file(s) loaded from ../Knowledge/

Total characters loaded: 3739


## 2. Generate Questions
This cell uses Gemini to generate a diverse set of questions based on the guidelines.

In [6]:
if not combined_text.strip():
    print("❌ No content to process. Please check your sources.")
else:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3, google_api_key = GEMINI_API)

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert in creating evaluation datasets for RAG systems. Your task is to generate a test set of questions from the provided information. Follow these guidelines:

{guidelines}
Generate questions in a JSON format, where each object has a 'question' and a 'ground_truth' field. The 'ground_truth' should be the detailed, correct answer to the question based on the provided text."""),
        ("human", """Generate 20 diverse questions in JSON format from the following text:

{text}""")
    ])

    parser = JsonOutputParser()
    chain = prompt | llm | parser

    try:
        print("Generating questions (this might take a moment depending on text size)...")
        # Truncate text if it's extremely large to stay within context limits (e.g. 100k chars)
        questions = chain.invoke({"guidelines": guidelines, "text": combined_text})
        
        # Save to file
        output_file = 'test_questions.json'
        with open(output_file, 'w') as f:
            json.dump(questions, f, indent=2)
        
        print(f"✅ Successfully generated {len(questions)} questions.")
        print(f"Saved to {output_file}")
        
    except Exception as e:
        print(f"❌ Error: {e}")

Generating questions (this might take a moment depending on text size)...
✅ Successfully generated 20 questions.
Saved to test_questions.json
